In [1]:
import numpy as np  

# Modulo Labo1

In [2]:
def error(x, y):
    x = np.float64(x)
    y = np.float64(y)
    
    error = abs(x - y)
    
    return error


In [3]:
def error_relativo(x, y):
    x = np.float64(x)
    y = np.float64(y)
    
    numerador = abs(x - y)
    denominador = abs(x)
    
    error_relativo = numerador/denominador
    
    return error_relativo

In [4]:
def matricesIguales(A, B):
    
    if len(A) != len(B) or len(A[0]) != len(B[0]):
        return False
    
    # Aca suponemos que las dimensiones son iguales
    for n in range (len(A)):
        # no necesariamente es cuadrada 
        for m in range(len(A[0])):
            # if A[n][m] != B[n][m]:
            # Con el distinto no me pasa por lo que lo panteo con error
            if abs (A[n][m] - B[n][m]) > 1e-08:
                return False
            
    return True

## Test


In [5]:
def sonIguales(x,y,atol=1e-08):
    return np.allclose(error(x,y),0,atol=atol)

assert(not sonIguales(1,1.1))
assert(sonIguales(1,1 + np.finfo('float64').eps))
assert(not sonIguales(1,1 + np.finfo('float32').eps))
assert(not sonIguales(np.float16(1),np.float16(1) + np.finfo('float32').eps))
assert(sonIguales(np.float16(1),np.float16(1) + np.finfo('float16').eps,atol=1e-3))

assert(np.allclose(error_relativo(1,1.1),0.1))
assert(np.allclose(error_relativo(2,1),0.5))
assert(np.allclose(error_relativo(-1,-1),0))
assert(np.allclose(error_relativo(1,-1),2))

assert(matricesIguales(np.diag([1,1]),np.eye(2)))
assert(matricesIguales(np.linalg.inv(np.array([[1,2],[3,4]]))@np.array([[1,2],[3,4]]),np.eye(2)))
assert(not matricesIguales(np.array([[1,2],[3,4]]).T,np.array([[1,2],[3,4]])))

# Modulo Labo 2

In [36]:
def rota(theta):
    # Para esta matriz tenemos que la forma es [[cos, sin], [-sin,cos]]
    # usamos las funciones trigonometricas de numpy
    
    R = np.array([[np.cos(theta), -np.sin(theta)], 
                 [np.sin(theta), np.cos(theta)]])
    
    return R

In [37]:
def escala(s):
    n = len(s)
    S = np.zeros((n,n))
    
    for i in range(len(s)):
        S[i][i] = s[i]
        
    return S

In [38]:
def multiplicacionMatricial(A,B):
    
    if len(A[0]) !=  len(B):
        return None
    
    res = np.zeros((len(A), len(B[0])), dtype = float)
    suma = 0
    
    for i in range(len(A)):
        for j in range(len(B[0])):
            # Este k puede iterar tanto en la len de las columans de A
            # como en la len de las filas de B
            for k in range(len(A[0])):
                elem_A = A[i][k]
                elem_B = B[k][j]
                suma += elem_A * elem_B
            
            res [i][j] = suma
            suma = 0
                  
    return res
      
                

In [39]:
def rota_y_escala(theta, s):
    # ojo que para que la matriz sea de 2*2, s tiene que tener len(s) = 2
    if len(s) != 2:
        return None
    R = rota(theta)
    S = escala(s)
    
    # Si primero rota y despues escala es RoS
    res = multiplicacionMatricial(S,R)
    
    return res
     
    

In [40]:
def afin(theta,s,b):
    if len(s) != 2 or len(b) != 2:
        return None
    
    R_aux = rota(theta)
    S_aux = escala(s)
    
    multiplicacion = multiplicacionMatricial(S_aux, R_aux)
    
    res = np.zeros((3,3))
    
    for i in range(2):
        for j in range(2):
            res[i][j] = multiplicacion[i][j]
            
    for i in range(len(b)):
        res[i][2] = b[i]
        
    res[2][2] = 1
    
    return res
    
     

In [41]:
def multiplicacionMatrizVector(A, v):
    
    if len(A[0]) != len(v):
        return None

    res = np.zeros(len(A))
    
    for i in range(len(A[0])):
        suma = 0
        for k in range(len(v)):
            suma += A[i][k] * v[k]
            
        res[i] = suma
        
    return res
        
        
            

In [42]:
A = np.array([[1,2,1],[1,1,0], [2,1,4]])
v = np.array([3,1,1])
multiplicacionMatrizVector(A, v)

array([ 6.,  4., 11.])

In [43]:
def trans_afin(v, theta, s, b):
    A = afin(theta, s, b)
    
    nuevo_v = np.zeros(3)
    
    for i in range(len(v)):
        nuevo_v[i] = v[i]
    nuevo_v[2] = 1
    A_v = multiplicacionMatrizVector(A, nuevo_v)
    
    res = A_v[:2]
    
    
    return res

## Test

In [44]:

import numpy as np
# from lab2 import rota, escala, rota_y_escala, afin, trans_afin


# --- Tests extraídos del enunciado ---

print("\nIniciando serie de tests del Laboratorio 2...")

# Tests para rota
assert(np.allclose(rota(0), np.eye(2)))
assert(np.allclose(rota(np.pi/2), np.array([[0, -1],[1, 0]])))
assert(np.allclose(rota(np.pi), np.array([[-1, 0],[0, -1]])))

# Tests para escala
assert(np.allclose(escala([2,3]), np.array([[2,0],[0,3]])))
assert(np.allclose(escala([1,1,1]), np.eye(3)))
assert(
    np.allclose(escala([0.5,0.25]), np.array([[0.5,0],[0,0.25]]))
)

# Tests para rota_y_escala
assert(
    np.allclose(rota_y_escala(0,[2,3]), np.array([[2,0],[0,3]]))
)
assert(np.allclose(
    rota_y_escala(np.pi/2,[1,1]), np.array([[0,-1],[1,0]])
))
assert(np.allclose(
    rota_y_escala(np.pi,[2,2]), np.array([[-2,0],[0,-2]])
))

# Tests para afin
assert(np.allclose(
    afin(0,[1,1],[1,2]),
    np.array([[1,0,1],
              [0,1,2],
              [0,0,1]])))
assert(np.allclose(afin(np.pi/2,[1,1],[0,0]),
    np.array([[0,-1,0],
              [1,0,0],
              [0,0,1]])))
assert(np.allclose(afin(0,[2,3],[1,1]),
    np.array([[2,0,1],
              [0,3,1],
              [0,0,1]])))

# Tests para trans_afina

assert(np.allclose(
    trans_afin(np.array([1,0]), np.pi/2,[1,1],[0,0]),
    np.array([0,1])
))
assert(np.allclose(
    trans_afin(np.array([1,1]), 0,[2,3],[0,0]),
    np.array([2,3])
))
assert(np.allclose(
    trans_afin(np.array([1,0]), np.pi/2,[3,2],[4,5]),
    np.array([4,7])
))

print("¡Todos los tests del Laboratorio 2 han pasado exitosamente!")


Iniciando serie de tests del Laboratorio 2...
¡Todos los tests del Laboratorio 2 han pasado exitosamente!


# Modulo Labo 3

In [2]:
def norma(x, p):
    # x es el vector y p es la norma
    for i in range(len(x)):
        x[i] = abs(x[i])
        
    if p == 'inf':
        return max(x)
    
    suma = 0
    for elem in x:
        suma += elem**p
    
    raiz = suma**(1/p)
    
    return raiz
    

In [3]:
def normaliza(X, p):
    # X es una lista de vectores
    Y = []
    for vector in X:
        norma_v = norma(vector, p)
        y = np.zeros(len(vector))
        for i in range(len(vector)):
            elem = vector[i]
            y[i] = elem/norma_v
            
        Y.append(y)
        
    return Y

In [ ]:
def normaMatMC(A, q, p, Np):

In [27]:
def normaExacta(A, p=[1, 'inf']):
    
    # inf
    suma_filas = []
    # 1
    suma_columnas = []
    
    for i in range(len(A)):
        suma = 0
        for j in range(len(A[0])):
            suma += abs(A[i][j])
            
        suma_filas.append(suma)
        
    for i in range(len(A[0])):
        suma = 0
        for j in range(len(A)):
            suma += abs(A[j][i])
            
        suma_columnas.append(suma)
    

    if type(p) == list:
        return [max(suma_columnas), max(suma_filas)]
    elif p == 1:  
        return max(suma_columnas)
    elif p == 'inf':
        return max(suma_filas)
    else:
        return None

In [ ]:
def condMC(A,p):
  inversa = np.linalg.inv(A)

  # Pongo 10000 porque es el valor que esta en los test
  # Es una multiplicacion de numeros no es matricial
  cond = normaMatMC(A,p,p,10000)[0] * normaMatMC(inversa, p, p, 10000)[0]

  return cond

In [ ]:
def condExacto(A,p):
  inversa = np.linalg.inv(A)

  cond = normaExacta(A,p) * normaExacta(inversa, p)

  return cond

In [23]:
A1 = np.array([
    [ 1, -2,  3],
    [-4,  5, -6]
])
type(A1)

numpy.ndarray

## Test


In [28]:
# Tests L03-Normas

# Tests norma
assert(np.allclose(norma(np.array([1,1]),2),np.sqrt(2)))
assert(np.allclose(norma(np.array([1]*10),2),np.sqrt(10)))
# assert(norma(np.random.rand(10),2)<=np.sqrt(10))
# assert(norma(np.random.rand(10),2)>=0)

# Tests normaliza
# Tests normaliza
for x in normaliza([np.array([1]*k) for k in range(1,11)],2):
    assert(np.allclose(norma(x,2),1))
for x in normaliza([np.array([1]*k) for k in range(2,11)],1):
    print(not np.allclose(norma(x,2),1) )
for x in normaliza([np.random.rand(k) for k in range(1,11)],'inf'):
    assert( np.allclose(norma(x,'inf'),1) )


# Tests normaExacta

assert(np.allclose(normaExacta(np.array([[1,-1],[-1,-1]]),1),2))
assert(np.allclose(normaExacta(np.array([[1,-2],[-3,-4]]),1),6))
assert(np.allclose(normaExacta(np.array([[1,-2],[-3,-4]]),'inf'),7))
assert(normaExacta(np.array([[1,-2],[-3,-4]]),2) is None)
assert(normaExacta(np.random.random((10,10)),1)<=10)
assert(normaExacta(np.random.random((4,4)),'inf')<=4)
'''
# Test normaMC

nMC = normaMatMC(A=np.eye(2),q=2,p=1,Np=100000)
assert(np.allclose(nMC[0],1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),0,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),0,atol=1e-3))

nMC = normaMatMC(A=np.eye(2),q=2,p='inf',Np=100000)
assert(np.allclose(nMC[0],np.sqrt(2),atol=1e-3))
assert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) and np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))

A = np.array([[1,2],[3,4]])
nMC = normaMatMC(A=A,q='inf',p='inf',Np=1000000)
assert(np.allclose(nMC[0],normaExacta(A,'inf'),rtol=2e-1)) 

# Test condMC

A = np.array([[1,1],[0,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

A = np.array([[3,2],[4,1]])
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaMatMC(A,2,2,10000)
normaA_ = normaMatMC(A_,2,2,10000)
condA = condMC(A,2,10000)
assert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))

# Test condExacta

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,1)
normaA_ = normaExacta(A_,1)
condA = condExacta(A,1)
assert(np.allclose(normaA*normaA_,condA))

A = np.random.rand(10,10)
A_ = np.linalg.solve(A,np.eye(A.shape[0]))
normaA = normaExacta(A,'inf')
normaA_ = normaExacta(A_,'inf')
condA = condExacta(A,'inf')
assert(np.allclose(normaA*normaA_,condA))
'''

True
True
True
True
True
True
True
True
True


"\n# Test normaMC\n\nnMC = normaMatMC(A=np.eye(2),q=2,p=1,Np=100000)\nassert(np.allclose(nMC[0],1,atol=1e-3))\nassert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))\nassert(np.allclose(np.abs(nMC[1][0]),0,atol=1e-3) or np.allclose(np.abs(nMC[1][1]),0,atol=1e-3))\n\nnMC = normaMatMC(A=np.eye(2),q=2,p='inf',Np=100000)\nassert(np.allclose(nMC[0],np.sqrt(2),atol=1e-3))\nassert(np.allclose(np.abs(nMC[1][0]),1,atol=1e-3) and np.allclose(np.abs(nMC[1][1]),1,atol=1e-3))\n\nA = np.array([[1,2],[3,4]])\nnMC = normaMatMC(A=A,q='inf',p='inf',Np=1000000)\nassert(np.allclose(nMC[0],normaExacta(A,'inf'),rtol=2e-1)) \n\n# Test condMC\n\nA = np.array([[1,1],[0,1]])\nA_ = np.linalg.solve(A,np.eye(A.shape[0]))\nnormaA = normaMatMC(A,2,2,10000)\nnormaA_ = normaMatMC(A_,2,2,10000)\ncondA = condMC(A,2,10000)\nassert(np.allclose(normaA[0]*normaA_[0],condA,atol=1e-3))\n\nA = np.array([[3,2],[4,1]])\nA_ = np.linalg.solve(A,np.eye(A.shape[0]))\nnormaA = normaMatMC(A,2,

# Modulo Labo 4

In [ ]:
def matricesIguales(A, B):

    if len(A) != len(B) or len(A[0]) != len(B[0]):
        return False

    # Aca suponemos que las dimensiones son iguales
    for n in range (len(A)):
        # no necesariamente es cuadrada
        for m in range(len(A[0])):
            # if A[n][m] != B[n][m]:
            # Con el distinto no me pasa por lo que lo panteo con error
            if abs (A[n][m] - B[n][m]) > 1e-03:
                return False

    return True

In [ ]:
def calculaLU(A): #Desde el vamos ya asumimos que A es una matriz en consecuencia como primer paso creo mi matriz L

    if A is None:
        return None, None, 0

    filas: int=A.shape[0]
    columnas: int=A.shape[1] #En la materia siempre usamos matrices cuadradas para esto por lo que esto esta un poco de mas
    if filas != columnas:
        return None, None, 0

    matriz_l= np.eye(filas) #Creo la matriz identidad que va ser mi L
    matriz_u= np.copy(A)
    nops=0

    columna: int=0 #Aca vamos a medir la medida en el cual el pivot va a trabajar sobre "x" columnas
    fila: int=1 #Lo mismo pero con las fila

    for i in range(len(A)):
        pivote = matriz_u[i][i]

        if pivote == 0:
            return None, None, 0

        for k in range(fila, len(A)):
            m = matriz_u[k][columna] / pivote #Buscamos el valor de m que por "definicion" es esta formulita
            nops += 1
            matriz_l[k][columna] = m #Cambiammos el valor en mi matriz L
            matriz_u[k][columna] = 0 #En mi nueva matriz U le asigno el valor 0
            for j in range(columna + 1, columnas): #Esto lo agregue de una forma nueva para contar las operaciones una por una ya que "no puedo restar arrays" ya q no se me computan las operacion
                matriz_u[k][j] = matriz_u[k][j] - m * matriz_u[fila-1][j]
                nops += 2

        fila += 1
        columna += 1
    return matriz_l, matriz_u, nops

In [ ]:
def res_tri(L,b, inferior = True):
    n = len(L)
    solucion = np.zeros(len(b))
    if inferior:
        for i in range(n):
            suma = 0
            for j in range(i):
                suma += L[i][j] * solucion[j]
            solucion[i] = (b[i] - suma) / L[i][i]

    else:
        for i in range(n - 1, -1, -1):#El Primer caso es cuando agarra la ultima fila de todas y asi va iterando
            suma = 0
            for j in range(i + 1, n): #Todo lo que esta dentro de este rango es la sumatoria
                suma += L[i][j] * solucion[j]
            solucion[i] = (b[i] - suma) / L[i][i]
    return solucion

In [ ]:
def inversa(A):
    filas: int=A.shape[0]
    matriz_identidad= np.eye(filas)
    matriz_l, matriz_u , nups = calculaLU(A)

    if matriz_l is None or matriz_u is None:
        return None

    matriz_final=np.zeros((filas,filas))

    for i in range(filas):
        b= np.zeros((filas))
        b[i]=1
        y_temporal=res_tri(matriz_l,b,inferior=True)
        x_temporal=res_tri(matriz_u,y_temporal,inferior=False)
        for j in range(filas):
            matriz_final[j][i]=x_temporal[j] #Fila i columna j
    return matriz_final

In [ ]:
def inversa_diagonal(A):
    filas = A.shape[0]
    matriz_B = np.zeros((filas, filas))

    for i in range(filas):
        if A[i][i] == 0:
            return None

        matriz_B[i][i] = 1 / A[i][i]

    return matriz_B

In [16]:
def multiplicar_matrices(A, B):

    filas_A = A.shape[0]
    columnas_B = B.shape[1]

    if A.shape[1] != B.shape[0]:
        return None

    matriz_final = np.zeros((filas_A, columnas_B))

    for i in range(filas_A):
        for j in range(columnas_B):
            suma = 0
            for k in range(A.shape[1]):
                suma += A[i][k] * B[k][j]
            matriz_final[i][j] = suma

    return matriz_final

In [ ]:
def calculaLDV(A):
    #A=LDV
    #U=DV
    #A=LU
    matriz_l, matriz_u, operaciones = calculaLU(A)
    if matriz_l is None or matriz_u is None:
        return None

    filas=len(A)
    matriz_d=np.zeros((filas,filas))
    for i in range(filas):
        matriz_d[i][i]=matriz_u[i][i]

    # Matriz L ya la tenemos
    # Caso en el cual U=DV =>  V=D*-1xU
    # D es la diagonal de U
    matriz_d_inv= inversa_diagonal(matriz_u)
    matriz_v= multiplicar_matrices(matriz_d_inv,matriz_u)

    return matriz_l, matriz_d, matriz_v

In [15]:
def traspuesta(A):

    filas = A.shape[0]
    columnas = A.shape[1]

    matriz_t = np.zeros((columnas, filas))

    for i in range(filas):
        for j in range(columnas):
            matriz_t[j][i] = A[i][j]

    return matriz_t

In [ ]:
def esSDP(A, atol =1e-10):
  # Tengo que verificar que A sea simetrica y que los elementos de D sean +
  # Basta con chequear que los elementos de la diagonal sean positivos porque
  # luego voy a tomar raiz de D y tienen que ser todos positivos o tambien
  # puede ser porque cuando hago las multiplicaciones entre L y D que son TRI-
  # ANGULARES el determinante siempre se va a quedar con elementos de la diagonal
  #A es SDP <=> A= A*T y la diagonal de D es posiutiva

    if not matricesIguales(A, traspuesta(A)):
        return False

    resultado = calculaLDV(A)

    if resultado is None:
        return False

    matriz_l, matriz_d, matriz_v = resultado

    for i in range(len(matriz_d)):
        if matriz_d[i][i] <= atol:
            return False

    return True

## Test


In [ ]:
import numpy as np

# TESTS L04-LU

# TESTS LU
print("TESTS calculaLU")

L0 = np.array([[1,0,0],
               [0,1,0],
               [1,1,1]])

U0 = np.array([[10,1,0],
               [0,2,1],
               [0,0,1]])

A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(np.allclose(L,L0))
assert(np.allclose(U,U0))


L0 = np.array([[1,0,0],
               [1,1.001,0],
               [1,1,1]])

U0 = np.array([[1,1,1],
               [0,1,1],
               [0,0,1]])
A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(not np.allclose(L,L0))
assert(not np.allclose(U,U0))
assert(np.allclose(L,L0,atol=1e-3))
assert(np.allclose(U,U0,atol=1e-3))
assert(nops == 13)

L0 = np.array([[1,0,0],
               [1,1,0],
               [1,1,1]])

U0 = np.array([[1,1,1],
               [0,0,1],
               [0,0,1]])

A =  L0 @ U0
L,U,nops = calculaLU(A)
assert(L is None)
assert(U is None)
assert(nops == 0)

assert(calculaLU(None) == (None, None, 0))

assert(calculaLU(np.array([[1,2,3],[4,5,6]])) == (None, None, 0))

print("-----ÉXITO!!!!\n")


## TESTS res_tri
print("TESTS res_tri")

A = np.array([[1,0,0],
              [1,1,0],
              [1,1,1]])

b = np.array([1,1,1])
assert(np.allclose(res_tri(A,b),np.array([1,0,0])))

b = np.array([0,1,0])
assert(np.allclose(res_tri(A,b),np.array([0,1,-1])))

b = np.array([-1,1,-1])
assert(np.allclose(res_tri(A,b),np.array([-1,2,-2])))

b = np.array([-1,1,-1])
assert(np.allclose(res_tri(A,b,inferior=False),np.array([-1,1,-1])))

A = np.array([[3,2,1],[0,2,1],[0,0,1]])
b = np.array([3,2,1])
assert(np.allclose(res_tri(A,b,inferior=False),np.array([1/3,1/2,1])))

A = np.array([[1,-1,1],[0,1,-1],[0,0,1]])
b = np.array([1,0,1])
assert(np.allclose(res_tri(A,b,inferior=False),np.array([1,1,1])))
print("-----ÉXITO!!!!\n")


# Test inversa
print("TESTS inversa")

def esSingular(A):
    try:
        np.linalg.inv(A)
        return False
    except:
        return True

# Por que no siempre es invertible, hacemos varios tests
ntest = 10
for i in range(ntest):
    A = np.random.random((4,4))
    A_ = inversa(A)
    if not esSingular(A):
        inversaConNumpy = np.linalg.inv(A)
        assert(A_ is not None)
        assert(np.allclose(inversaConNumpy,A_))
    else: 
        assert(A_ is None)

# Matriz singular devería devolver None
A = np.array([[1,2,3],[4,5,6],[7,8,9]])
assert(inversa(A) is None)

print("-----ÉXITO!!!!\n")



# Test LDV:
print("TESTS calculaLDV")

L0 = np.array([[1,0,0],[1,1.,0],[1,1,1]])
D0 = np.diag([1,2,3])
V0 = np.array([[1,1,1],[0,1,1],[0,0,1]])
A =  L0 @ D0 @ V0
L,D,V = calculaLDV(A)
assert(np.allclose(L,L0))
assert(np.allclose(D,D0))
assert(np.allclose(V,V0))


L0 = np.array([[1,0,0],[1,1.001,0],[1,1,1]])
D0 = np.diag([3,2,1])
V0 = np.array([[1,1,1],[0,1,1],[0,0,1.001]])
A =  L0 @ D0  @ V0
L,D,V = calculaLDV(A)
assert(np.allclose(L,L0,1e-3))
assert(np.allclose(D,D0,1e-3))
assert(np.allclose(V,V0,1e-3))

print("-----ÉXITO!!!!\n")

# TESTS SDP
print("TESTS esSDP")

L0 = np.array([[1,0,0],[1,1,0],[1,1,1]])
D0 = np.diag([1,1,1])
A = L0 @ D0 @ L0.T
assert(esSDP(A))

D0 = np.diag([1,-1,1])
A = L0 @ D0 @ L0.T
assert(not esSDP(A))

D0 = np.diag([1,1,1e-16])
A = L0 @ D0 @ L0.T
assert(not esSDP(A))

L0 = np.array([[1,0,0],
               [1,1,0],
               [1,1,1]])
D0 = np.diag([1,1,1])
V0 = np.array([[1,0,0],
               [1,1,0],
               [1,1+1e-3,1]]).T
A = L0 @ D0 @ V0
assert(esSDP(A,1e-3))

print("-----ÉXITO!!!!\n")
print("---FINALIZADO LABO 4!---")

# Modulo Labo 5

Entonces se pueden tener factorizaciones QR por dos vias. Una es la de G-S y otra es la de HH

In [ ]:
def QR_con_GS(A, tol=1e-12, retorna_nops=False):
    
    # Para el caso que no es cuadrada
    if np.shape(A)[0] != np.shape(A)[1]:
        return None 
    #   return None, None, None?
    
    n = len(A)
    Q = np.zeros((n,n))
    R = np.zeros((n,n))
    nops = 0
    
    # Para tener las columnas mas comodo, hacemos la traspuesta
    # y nos vamos moviendo con los indices A.T[0]...A.T[n]
    A_t = traspuesta(A)
    
    # norma esta implementada en el labo3
    Q[0] = A_t[0] / norma(A_t[0], 2)
    nops += 1
    
    R[0][0] = norma(A_t[0], 2)
                   # vector
    
    for j in range(1,n):
        
        q_monio = A_t[j]
        
        for k in range(1, j - 1):
            # Hay que contar cada multiplicacion y la suma
            # Podemos suponer que son n, no?
            R[k][j] = Q[k] @ q_monio
                    # vector por vector
            nops += n    + n - 1
                  # mult + sumas
                    
            # ortogonalizo
            # no dividimos por la norma porque es 1
            q_monio = q_monio - R[k][j] * Q[k]
                                # escalar por vector
            
            nops += n      + n
                  # resta  + mult por escalar
        
        # No deberia ser al cuadrado la norma?
        R[j][j] = norma(q_monio, 2)
        # cuando hago la resta de arriba, el vector resultante
        # podria no tener norma 1 por eso lo normalizo.
        Q[j] = q_monio / R[j][j]
        nops += n
              # division por cada elemento de q_monio
              
    # A lo ultimo hay que trasponer la Q que te queda
    # porque estamos pasando todo como filas
    Q = traspuesta(Q)
    
    # Dice que retorna opcionalmente la cantidad de operaciones
    # Si False es que nos las tenes que devolver
    # Si True las tenes que devolver
    
    if retorna_nops == True:
        return Q, R, nops

    # el caso False
    return Q, R
    

In [ ]:
def QR_con_HH(A, tol=1e-12, extras = False):
    
    if np.shape(A)[0] < np.shape(A)[1]:
        return None
    
    
    # Inicializo
    m = np.shape(A)[0]
    n = np.shape(A)[1]
    R = A
    Q = np.eye(np.shape(A)[0])
    I_k = np.eye(m)
    extra_info = {
        'R_matrices': [],
        'Q_matrices': []
    }
    
    for k in range(n - 1):
        
        # Agarro el vector. Esta notacion dice agarro las
        # filas k hasta m y de esas solo tomo la columna k
        x = R[k:m, k]
        alpha = - np.sign(x[0]) * norma(x, 2)
        y = alpha * I_k[k:m, k]
        u = x - y
        norma_u = norma(u, 2)
        if norma_u > tol:
            # normalizo
            u = u / norma_u
            # * es elemento a elemento y @ es prod interno/ mult matricial
            H_k = I_k[k:m, k:n] - 2 * (u @ u) # -> IMPLEMENTAR @
            
            I_aux = np.eye(m)
            I_aux[k:m, k:m] = H_k
            H_k_monio = I_aux 
            
            R = H_k_monio @ R
            extra_info['R_matrices'].append(R)
            # H_k_monio es igual a la traspuesta 
            Q = Q @ H_k_monio
            extra_info['Q_matrices'].append(Q)
    
    if extras == False:
        return Q, R
    if extras == True:
        return Q, R, extra_info

In [35]:
A = np.array([[1,2],
              [2,1], 
              [2,4]])
print(A[0:3, 0])
print(np.eye(4,4))


[1 2 2]
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [36]:
I_aux = np.eye(3)
I_aux[1:3, 1:3] = np.array([[1,2],[3,2]])
H_k_monio = I_aux 
print(H_k_monio)

[[1. 0. 0.]
 [0. 1. 2.]
 [0. 3. 2.]]


## Tests

In [ ]:
### Funciones L05-QR
def QR_con_GS(A,tol=1e-12,retorna_nops=False):
    """
    A una matriz de n x n 
    tol la tolerancia con la que se filtran elementos nulos en R
    retorna_nops permite (opcionalmente) retornar el numero de operaciones realizado
    retorna matrices Q y R calculadas con Gram Schmidt (y como tercer argumento opcional, el numero de operaciones).
    Si la matriz A no es de n x n, debe retornar None
    """

def QR_con_HH(A,tol=1e-12,extras=False):
    """
    A una matriz de m x n (m>=n)
    tol la tolerancia con la que se filtran elementos nulos en R
    retorna matrices Q y R calculadas con reflexiones de Householder
    Si la matriz A no cumple m>=n, debe retornar None
    extras : bool, opcional
        Si es True, devuelve informacion extra sobre el proceso de factorizacion.
        Por defecto es False. Esto lo hacemos para poder graficar el proceso.
    Devuelve la factorizacion QR de A usando reflectores de Householder.
    Devuelve: 
        Q, R, extra_info (si extras es True)
        Q, R (si extras es False)
    extra_info es un diccionario con la clave:
        'R_matrices': lista de las matrices R en cada paso
        'Q_matrices': lista de las matrices Q en cada paso


    """
def calculaQR(A,metodo='RH',tol=1e-12):
    """
    A una matriz de n x n 
    tol la tolerancia con la que se filtran elementos nulos en R    
    metodo = ['RH','GS'] usa reflectores de Householder (RH) o Gram Schmidt (GS) para realizar la factorizacion
    retorna matrices Q y R calculadas con Gram Schmidt (y como tercer argumento opcional, el numero de operaciones)
    Si el metodo no esta entre las opciones, retorna None
    """

# Tests L05-QR:

import numpy as np

# --- Matrices de prueba ---
A2 = np.array([[1., 2.],
               [3., 4.]])

A3 = np.array([[1., 0., 1.],
               [0., 1., 1.],
               [1., 1., 0.]])

A4 = np.array([[2., 0., 1., 3.],
               [0., 1., 4., 1.],
               [1., 0., 2., 0.],
               [3., 1., 0., 2.]])

# --- Funciones auxiliares para los tests ---
def check_QR(Q,R,A,tol=1e-10):
    # Comprueba ortogonalidad y reconstrucción
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=tol)
    assert np.allclose(Q @ R, A, atol=tol)

# --- TESTS PARA QR_by_GS2 ---
Q2,R2 = QR_con_GS(A2)
check_QR(Q2,R2,A2)

Q3,R3 = QR_con_GS(A3)
check_QR(Q3,R3,A3)

Q4,R4 = QR_con_GS(A4)
check_QR(Q4,R4,A4)

# --- TESTS PARA QR_by_HH ---
Q2h,R2h = QR_con_GS(A2)
check_QR(Q2h,R2h,A2)

Q3h,R3h = QR_con_HH(A3)
check_QR(Q3h,R3h,A3)

Q4h,R4h = QR_con_HH(A4)
check_QR(Q4h,R4h,A4)

# --- TESTS PARA calculaQR ---
Q2c,R2c = calculaQR(A2,metodo='RH')
check_QR(Q2c,R2c,A2)

Q3c,R3c = calculaQR(A3,metodo='GS')
check_QR(Q3c,R3c,A3)

Q4c,R4c = calculaQR(A4,metodo='RH')
check_QR(Q4c,R4c,A4)

# Modulo Labo 6

In [30]:
def multiplicacionMatricesVectores(A, v):
    
    filas_A, columnas_A = np.shape(A)
    filas_v = np.shape(v)[0]
    
    if filas_v != columnas_A:
        raise ValueError('El tamano de las columnas de A no coincide con el tamano de las filas de v') 
    
    res = np.zeros(filas_A)
    
    for fila in range(filas_A):
        suma = 0
        for columna in range(columnas_A):
            suma += A[fila][columna] * v[columna]
        res[fila] = suma
        
    return res

In [33]:
def multiplicacionVectoresVectores(a, b):
    # Los vectores deben de tener la misma longitud
    
    if np.shape(a)[0] != np.shape(b)[0]:
        raise ValueError('Los vectores tienen diferente tamano')
    
    suma = 0
    for i in range(np.shape(a)[0]):
        suma += a[i] * b[i]
        
    return suma 

In [36]:
def metpot2k(A, tol = 1e-15, K = 1000):
    # es f^2 porque dice metodo de pot 2k!!!!
    
    n = np.shape(A)[0]
    # hacemos asi para que nos queden naturales random hasta el 10
    v = np.random.rand(n)
    v_mono = multiplicacionMatricesVectores(A, multiplicacionMatricesVectores(A, v))
    e = multiplicacionVectoresVectores(v_mono, v)
    k = 0
    
    while abs(e - 1) > tol and k < K:
        v = v_mono
        v_mono = multiplicacionMatricesVectores(A, multiplicacionMatricesVectores(A, v))
        e = multiplicacionVectoresVectores(v_mono, v)
        k = k + 1
        
    lamb = multiplicacionVectoresVectores(v_mono, multiplicacionMatricesVectores(A, v_mono))
    error = e - 1 
    
    return v, lamb, k, error

In [ ]:
def normaAlCuadrado(v):
    
    norma_al_cuadrado = 0
    for elem in v:
        norma_al_cuadrado += (abs(elem))**2
    return norma_al_cuadrado

In [9]:
def diagRH(A, tol=1e-15, K=1000):
    
    v_1, lamb_1, k, error = metpot2k(A, tol, K)
    n = np.shape(A)[0]
    I_n = np.eye(n)
    e_1 = I_n[0] 
    H_v_1 = I_n - 2(
        multiplicacionVectoresVectores((e_1 - v_1),(e_1 - v_1)) /
        normaAlCuadrado((e_1 - v_1))
    )
    
    if n == 2:
        S = H_v_1
        D = multiplicacionMatricial(H_v_1, multiplicacionMatricial(A, traspuesta(H_v_1)))
    else:
        B = multiplicacionMatricial(H_v_1, multiplicacionMatricial(A, traspuesta(H_v_1)))
        A_mono = B[1:n, 1:n]
        S_mono, D_mono = diagRH(A_mono, tol, K)
        
        # Armenmos D
        I_aux = np.eye(n)
        D, S_aux = I_aux
        D[0][0] = lamb_1
        D[1:n, 1:n] = D_mono
        S[1:n, 1:n] = S_aux
        S = multiplicacionMatricial(H_v_1, S_mono)

<>:7: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<>:7: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_15088/3688695128.py:7: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
  H_v_1 = I_n - 2(


In [8]:
I = np.eye(3)
print(I[0])

B = np.array([[1,1,1,1],
          [2,2,2,2],
          [3,1,3,3],
          [4,1,4,4]])

print(B[1:4, 1:4])

[1. 0. 0.]
[[2 2 2]
 [1 3 3]
 [1 4 4]]


## Test

In [ ]:
# Test L06-metpot2k, Aval

import numpy as np

#### TESTEOS
# Tests metpot2k

S = np.vstack([
    np.array([2,1,0])/np.sqrt(5),
    np.array([-1,2,5])/np.sqrt(30),
    np.array([1,-2,1])/np.sqrt(6)
              ]).T

# Pedimos que pase el 95% de los casos
exitos = 0
for i in range(100):
    D = np.diag(np.random.random(3)+1)*100
    A = S@D@S.T
    v,l,_ = metpot2k(A,1e-15,1e5)
    if np.abs(l - np.max(D))< 1e-8:
        exitos += 1
assert exitos > 95


#Test con HH
exitos = 0
for i in range(100):
    v = np.random.rand(9)
    #v = np.abs(v)
    #v = (-1) * v
    ixv = np.argsort(-np.abs(v))
    D = np.diag(v[ixv])
    I = np.eye(9)
    H = I - 2*np.outer(v.T, v)/(np.linalg.norm(v)**2)   #matriz de HouseHolder

    A = H@D@H.T
    v,l,_ = metpot2k(A, 1e-15, 1e5)
    #max_eigen = abs(D[0][0])
    if abs(l - D[0,0]) < 1e-8:         
        exitos +=1
assert exitos > 95



# Tests diagRH
D = np.diag([1,0.5,0.25])
S = np.vstack([
    np.array([1,-1,1])/np.sqrt(3),
    np.array([1,1,0])/np.sqrt(2),
    np.array([1,-1,-2])/np.sqrt(6)
              ]).T

A = S@D@S.T
SRH,DRH = diagRH(A,tol=1e-15,K=1e5)
assert np.allclose(D,DRH)
assert np.allclose(np.abs(S.T@SRH),np.eye(A.shape[0]),atol=1e-7)



# Pedimos que pase el 95% de los casos
exitos = 0
for i in range(100):
    A = np.random.random((5,5))
    A = 0.5*(A+A.T)
    S,D = diagRH(A,tol=1e-15,K=1e5)
    ARH = S@D@S.T
    e = normaExacta(ARH-A,p='inf')
    if e < 1e-5: 
        exitos += 1
assert exitos >= 95


/tmp/ipykernel_30335/1963385421.py:9: RuntimeWarning: overflow encountered in scalar multiply
  suma += a[i] * b[i]
/tmp/ipykernel_30335/1546151545.py:14: RuntimeWarning: overflow encountered in scalar multiply
  suma += A[fila][columna] * v[columna]


ValueError: too many values to unpack (expected 3)

# Modulo Labo 7

In [ ]:
def transiciones_al_azar_continuas(n):
    
    M = np.random.rand(n,n)
    sumas_col = np.zeros(n)
    
    for col in range(n):
        suma_col = 0
        for fila in range(n):
            suma_col += M[fila][col]
        
        sumas_col[col] = suma_col
    
    for col in range(n):
        if sumas_col[col] == 0:
            continue
        for fila in range(n):
            M[fila][col] = M[fila][col]/sumas_col[col]
    
    return M
    

In [ ]:
def tansicion_al_azar_uniforme(n, thres):
    
    M = np.random.rand(n, n)
    sumas_col = np.zeros(n)
    
    for i in range(n):
        for j in range(n):
            # el igual lo agrego yo. no esta en el enunciado
            if M[i][j] >= thres:
                M[i][j] = 0
            else:
                M[i][j] = 1
                
    for col in range(n):
        suma_col = 0
        for fila in range(n):
            suma_col += M[fila][col]
        
        sumas_col[col] = suma_col
        
    for col in range(n):
        if sumas_col[col] == 0:
            continue
        for fila in range(n):
            M[fila][col] = M[fila][col]/sumas_col[col]
    
    return M
            

In [3]:
A = np.random.rand(3,3)
print(A)

[[0.32145042 0.37211433 0.93554301]
 [0.81607403 0.12904058 0.3939829 ]
 [0.53549057 0.20029523 0.19768112]]



np.cos()


np.sen()


np.eye()


np.shape()


np.zeros()


np.copy()


np.ones()


np.ndim()


np.arrange()


np.linspace()


np.array()


np.reshape()


Funciones del submódulo np.random que sirvan para generar números pseudo-aleatorios.


Operaciones de slicing



In [ ]:
def miBsAsCrecido(alto, ancho):
    n = len(alto)
    
    M = [[-1] * n for _ in range(n)]
    
    for i in range(n):
        M[i][n] = 0
        
    for act in range(n - 1, -1):
        for ult in range(n - 1, -1):
            if alto[ult] < alto[act]:
                M[act][ult] = M[act + 1][ult]
                
            M[act][ult] = max(M[act + 1][ult], M[act + 1][act] + ancho[act])
            
    return M[0][n]
            
        
    

In [ ]:
def maxSumaG(S,C,k):
    
    if len(S) == k:
        return S, sum(S)
    
    maximo_c = max(C)
    C.remove(maximo_c)
    S = S.add(maximo_c)
    
    return maxSumaG(S, C, k)